In [0]:
#sickness_absence_region_monthly code outline 
 
 
from pyspark.sql.functions import (col,trim,when,to_date,lit,current_timestamp,lower, substring, when) 


bronze_sickness_absence_region_monthly_df= ( 
	spark.read 
	.option("header", True) 
	.option("inferSchema", True) 
	.csv("abfss://data@jdnhsbronze.dfs.core.windows.net/dbo.jdnhs_sickness_absence_region_monthly.csv")
	) 
 
bronze_sickness_absence_region_monthly_df.show(10) 
bronze_sickness_absence_region_monthly_df.printSchema() 

# check schema here and if its wrong add code to make it correct data types 
#month already done 


In [0]:




# rename columns
bronze_sickness_absence_region_monthly_df = ( 
	bronze_sickness_absence_region_monthly_df  
	.withColumnRenamed("sickness_absence_rate_pct", "sick_leave_rate") )
	 
#standardise data
silver_sickness_absence_region_monthly_df = (bronze_sickness_absence_region_monthly_df 
	.withColumn(
        "month",to_date(
            substring(trim(col("month")), 1, 10),"yyyy-MM-dd"))
	.withColumn("region_name", lower(trim(col("region_name"))))
	.withColumn("source_file", lower(trim(col("source_file"))))
	.withColumn("source_sheet", lower(trim(col("source_sheet"))))
	) 

# clean data

silver_sickness_absence_region_monthly_df = (silver_sickness_absence_region_monthly_df 
    .withColumn("month",to_date(col("month"),"yyyy-MM-dd")) 
	.withColumn(
    "region_name",
    when(col("region_name") == "south west of england", "south west")
    .when(col("region_name") == "south east of england", "south east")
	.otherwise(col("region_name"))
 ))
 
# check data
silver_sickness_absence_region_monthly_df.show(10) 





In [0]:

# validate data
valid_sickness_absence_region_monthly_df = (silver_sickness_absence_region_monthly_df.filter( 
    col("month").isNotNull() & 
    col("region_name").isNotNull() &
    (col("region_name") != "special health authorities and other statutory bodies") &
    ( 
        col("sick_leave_rate").isNull() | 
        (col("sick_leave_rate").between(0, 100)) 
    ) 
))
 
 
quarantine_sickness_absence_region_monthly_df = silver_sickness_absence_region_monthly_df.filter( 
    col("month").isNull() | 
    col("region_name").isNull() | 
    (col("region_name") == "special health authorities and other statutory bodies") |
    ( 
        col("sick_leave_rate").isNotNull() & 
        (~col("sick_leave_rate").between(0, 100)) 
    ) 
) 


#check data
print(quarantine_sickness_absence_region_monthly_df.count())
print(valid_sickness_absence_region_monthly_df.count())






In [0]:
valid_sickness_absence_region_monthly_df.select("region_name").distinct().show(truncate=False)

In [0]:
quarantine_sickness_absence_region_monthly_df.show(10)
valid_sickness_absence_region_monthly_df.show(10)

In [0]:

# write data 



(valid_sickness_absence_region_monthly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://silver-tables@jdnhsbronze.dfs.core.windows.net/silver_sick_leave/"
    ) \
    .saveAsTable(
        "silver_sick_leave"
    ))

# Invalid data → Quarantine
(
    quarantine_sickness_absence_region_monthly_df.write
    .format("delta")
    .mode("overwrite")
    .save(
        "abfss://quarantine@jdnhsbronze.dfs.core.windows.net/silver_sick_leave/"
    )
)

